# Day 6 · Lab 2 — BRD Generation + HITL Sign-off

## What you'll build

1. Load requirements extracted from Lab 1
2. **Render a BRD** (Business Requirements Document) from a template
3. Build a **LangGraph agent** with 3 nodes: render_draft → HITL_pause → freeze_or_revise
4. Use **`interrupt_before`** for reviewer sign-off (same pattern as Day 1)
5. Inject reviewer decision via **`graph.update_state()`**
6. Resume with **`graph.invoke(None, config)`**
7. Test both approve and revise flows

## Prerequisites

- Lab 1 completed — /tmp/day6_requirements.json exists

## Step 1 — Environment + load requirements from Lab 1

In [ ]:
import os, sys, subprocess, json
from pathlib import Path

for pkg in ["python-dotenv", "langchain-openai", "langgraph"]:
    try:
        __import__(pkg.replace("-", "_").split("[")[0])
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--user", pkg])

from dotenv import load_dotenv
load_dotenv(Path("~/agentic-lab/.env").expanduser(), override=False)
for k in ("ANTHROPIC_API_KEY","OPENAI_API_KEY","LANGSMITH_API_KEY"):
    if os.environ.get(k) == "":
        del os.environ[k]

assert os.environ.get("OPENROUTER_API_KEY"), "OPENROUTER_API_KEY missing"
os.environ["OPENAI_API_KEY"] = os.environ["OPENROUTER_API_KEY"]
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

# Load requirements
req_path = Path("/tmp/day6_requirements.json")
assert req_path.exists(), "Run Lab 1 Step 10 first — /tmp/day6_requirements.json missing"

requirements = json.loads(req_path.read_text())
print(f"✓ Loaded {len(requirements)} requirements from Lab 1")

## Step 2 — BRD template + rendering functions

In [ ]:
BRD_TEMPLATE = '''# Business Requirements Document

**Project:** Customer Self-Service Portal — Phase 2
**Draft Version:** {version}
**Extracted from:** Product Planning Meeting
**Status:** {status}

## 1. Executive Overview

{overview}

## 2. Business Requirements

{business_table}

## 3. Functional Requirements

{functional_table}

## 4. Non-Functional Requirements

{nfr_table}

## 5. Acceptance Criteria

{acceptance_list}

## 6. Traceability

Every requirement in this document is traced to a verbatim source utterance from the meeting transcript. See Appendix A for full source mapping.

---
_Generated by Requirements Extraction Agent — pending sign-off_
'''


def format_req_table(reqs: list[dict]) -> str:
    if not reqs:
        return "_(none extracted)_"
    lines = ["| ID | Priority | Statement | Source Speaker |", "|---|---|---|---|"]
    for r in reqs:
        stmt = r['statement'].replace('|', '\\|')
        lines.append(f"| {r['id']} | {r['priority']} | {stmt} | {r['source_speaker']} |")
    return "\n".join(lines)


def format_acceptance_list(reqs: list[dict]) -> str:
    accs = [r for r in reqs if r['type'] == 'acceptance']
    if not accs:
        return "_(none extracted)_"
    return "\n".join(f"- **{r['id']}**: {r['statement']}" for r in accs)


print("✓ BRD template + renderers defined")

## Step 3 — Generate the overview with LLM

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="anthropic/claude-sonnet-4.5",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=0,
)


def generate_overview(reqs: list[dict]) -> str:
    business_reqs = [r for r in reqs if r['type'] == 'business']
    all_statements = "\n".join(f"- {r['statement']}" for r in reqs[:10])
    
    prompt = f"""Write a 2-paragraph executive overview for a Business Requirements Document.

Requirements captured:
{all_statements}

Focus on: business drivers, scope, and success criteria. Do NOT list individual requirements — they appear in later sections. Keep to 4-6 sentences total."""
    
    return llm.invoke(prompt).content.strip()


overview = generate_overview(requirements)
print("Generated overview:")
print(overview)

## Step 4 — Render the full BRD

In [ ]:
def render_brd(reqs: list[dict], version: str = "0.1", status: str = "DRAFT — awaiting sign-off") -> str:
    business = [r for r in reqs if r['type'] == 'business']
    functional = [r for r in reqs if r['type'] == 'functional']
    nfr = [r for r in reqs if r['type'] == 'non_functional']
    
    return BRD_TEMPLATE.format(
        version=version,
        status=status,
        overview=generate_overview(reqs),
        business_table=format_req_table(business),
        functional_table=format_req_table(functional),
        nfr_table=format_req_table(nfr),
        acceptance_list=format_acceptance_list(reqs),
    )


brd = render_brd(requirements)
print(brd[:1500])
print("\n...")
print(brd[-500:])

## Step 5 — LangGraph agent with HITL sign-off

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver


class SignoffState(TypedDict):
    requirements: list
    brd_draft: str
    approval: dict          # {status, reviewer, timestamp, feedback}
    final_brd: str
    version: str


def render_node(state: SignoffState) -> dict:
    version = state.get("version", "0.1")
    brd = render_brd(state["requirements"], version=version)
    return {"brd_draft": brd}


def signoff_node(state: SignoffState) -> dict:
    # This node runs AFTER interrupt — user has already injected approval
    return {}


def freeze_node(state: SignoffState) -> dict:
    approval = state.get("approval", {})
    status = approval.get("status", "unknown")
    
    if status == "approved":
        final = state["brd_draft"].replace(
            "DRAFT — awaiting sign-off",
            f"APPROVED by {approval.get('reviewer')} at {approval.get('timestamp')}"
        )
        return {"final_brd": final}
    elif status == "rejected":
        return {"final_brd": "[REJECTED — see approval.feedback]"}
    else:  # revise
        return {"final_brd": "[REVISE — regenerating draft]"}


def route_after_signoff(state: SignoffState) -> str:
    status = state.get("approval", {}).get("status", "")
    if status == "approved":
        return "freeze"
    if status == "revise":
        return "render"   # loop back
    return "freeze"   # reject also goes to freeze (records reason)


builder = StateGraph(SignoffState)
builder.add_node("render", render_node)
builder.add_node("signoff", signoff_node)
builder.add_node("freeze", freeze_node)

builder.add_edge(START, "render")
builder.add_edge("render", "signoff")
builder.add_conditional_edges("signoff", route_after_signoff,
                              {"freeze": "freeze", "render": "render"})
builder.add_edge("freeze", END)

memory = MemorySaver()
graph = builder.compile(
    checkpointer=memory,
    interrupt_before=["signoff"],   # pause for reviewer
)

print("✓ Sign-off agent compiled with interrupt_before=['signoff']")

## Step 6 — Run the agent, watch it pause at HITL

In [ ]:
import uuid

thread_id = f"brd-{uuid.uuid4().hex[:8]}"
config = {"configurable": {"thread_id": thread_id}}

initial_state = {
    "requirements": requirements,
    "brd_draft": "",
    "approval": {},
    "final_brd": "",
    "version": "0.1",
}

# First invocation — pauses at signoff
result = graph.invoke(initial_state, config)
print(f"Agent paused. Current state:")
state = graph.get_state(config)
print(f"  Next step: {state.next}")
print(f"  BRD draft length: {len(result.get('brd_draft', ''))}")
print(f"  Final BRD: '{result.get('final_brd', '(empty — waiting for sign-off)')}'")

## Step 7 — Reviewer approves — inject decision and resume

In [ ]:
from datetime import datetime

# Reviewer decision
graph.update_state(config, {
    "approval": {
        "status": "approved",
        "reviewer": "PO_sarah",
        "timestamp": datetime.utcnow().isoformat(),
        "feedback": "Requirements look complete. Approved for engineering handoff."
    }
})

# Resume — critical pattern: invoke with None
final = graph.invoke(None, config)
print(f"After approval:")
print(f"  Final BRD status line: {final['final_brd'][:200]}")
print(f"\n  Approval record: {final['approval']}")

## Step 8 — Try the REVISE flow

Different reviewer wants changes. Agent should loop back to render.

In [ ]:
thread_id_2 = f"brd-{uuid.uuid4().hex[:8]}"
config_2 = {"configurable": {"thread_id": thread_id_2}}

# Start fresh
result = graph.invoke({
    "requirements": requirements,
    "brd_draft": "",
    "approval": {},
    "final_brd": "",
    "version": "0.1",
}, config_2)

# Reviewer wants revisions
graph.update_state(config_2, {
    "approval": {
        "status": "revise",
        "reviewer": "TL_priya",
        "timestamp": datetime.utcnow().isoformat(),
        "feedback": "NFR section needs latency percentiles for all API endpoints, not just order lookup."
    },
    "version": "0.2",
})

# Resume — should loop back to render, then pause at signoff AGAIN
result = graph.invoke(None, config_2)
state = graph.get_state(config_2)
print(f"After revise-request:")
print(f"  Version: {result.get('version')}")
print(f"  Next step: {state.next}")   # should be signoff again
print(f"  Draft was re-rendered")
print(f"  Final BRD: '{result.get('final_brd')}'")

## Step 9 — Approve the revised version

In [ ]:
graph.update_state(config_2, {
    "approval": {
        "status": "approved",
        "reviewer": "TL_priya",
        "timestamp": datetime.utcnow().isoformat(),
        "feedback": "Second pass looks good."
    }
})

final = graph.invoke(None, config_2)
print(f"After second approval:")
print(f"  Version: {final['version']}")
print(f"  Final BRD (first 300 chars):")
print(final['final_brd'][:300])

## Step 10 — Audit trail

Every state change is checkpointed. You can walk the history for compliance audit.

In [ ]:
history = list(graph.get_state_history(config_2))
print(f"State transitions for thread {thread_id_2}:")
print(f"Total checkpoints: {len(history)}\n")

for i, snapshot in enumerate(reversed(history)):
    step = snapshot.next[0] if snapshot.next else "END"
    approval = snapshot.values.get("approval", {})
    version = snapshot.values.get("version", "?")
    approval_summary = f"{approval.get('status', 'pending')} by {approval.get('reviewer', '-')}" if approval else "no approval yet"
    print(f"  Step {i}: next={step}, v={version}, approval={approval_summary}")

## What you learned

1. **Template + LLM overview** — templates are code, LLM only fills narrative
2. **LangGraph HITL sign-off** — same `interrupt_before` + `invoke(None, config)` pattern from Day 1
3. **Approval as state** — reviewer decision + timestamp + user_id lives in state, becomes the audit record
4. **Loop-back for revise** — conditional edges route revise decisions back to render
5. **Checkpoint history IS the audit trail** — every state transition preserved for compliance

## Production notes

- Store checkpoints in PostgresSaver (not MemorySaver) for durable audit
- Add MCP tools to notify reviewer via Slack/email when HITL pauses
- Wire final BRD to Confluence upload + JIRA ticket creation
- Multi-reviewer: chain multiple interrupt_before nodes, one per role
- Sign each version cryptographically for regulated industries

## Day 6 complete

You've built:
- Extraction pipeline with typed output + traceability
- Document generation from templates
- HITL sign-off with full audit trail

Same LangGraph patterns. New application. Ready for real BA workflows.

Day 7: insight synthesis & reporting agents.
